DATA INGESTION

In [1]:
#document data structure
from langchain_core.documents import Document

c:\Users\owner\Desktop\python\RAG\rag-1\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
doc = Document(
    page_content='this is the main peice of the content we want to use for RAG',
    metadata={
        'source': 'example.txt',
        'pages': 1,
        'author': 'Kunle Ade',
        'date_created': '2025-01-01'
    }
)

doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Kunle Ade', 'date_created': '2025-01-01'}, page_content='this is the main peice of the content we want to use for RAG')

In [3]:
## create a simple txt file
import os
os.makedirs('../data/text_files', exist_ok=True) 

In [4]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("Sample text files created!")

Sample text files created!


In [5]:
##Text Loader

from langchain_community.document_loaders import TextLoader
loader = TextLoader('../data/text_files/python_intro.txt', encoding='utf-8')
document = loader.load()
print(document)

c:\Users\owner\Desktop\python\RAG\rag-1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [6]:
#Directory Loader 
from langchain_community.document_loaders import DirectoryLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt", ## Pattern to match files  
    loader_cls= TextLoader, ##loader class to use  # we even make a list for the class to include pdfs etc
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False )

documents=dir_loader.load()
documents

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    '),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popu

In [7]:
#Directory Loader 
###only one folder 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader 
## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf", ## Pattern to match files  
    loader_cls= PyMuPDFLoader, ##loader class to use  # we even make a list for the class to include pdfs etc
    show_progress=False )

pdf_Doc = dir_loader.load()
pdf_Doc


[Document(metadata={'producer': 'mPDF 7.1.5', 'creator': '', 'creationdate': "20181019192124+00'00'", 'source': '..\\data\\pdf\\cheatsheet.pdf', 'file_path': '..\\data\\pdf\\cheatsheet.pdf', 'total_pages': 3, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': "20181019192124+00'00'", 'trapped': '', 'modDate': "20181019192124+00'00'", 'creationDate': "20181019192124+00'00'", 'page': 0}, page_content="Interview Cheat Sheet\nFrom Andrei Neagoie's Master The Coding Interview: Data Structures + Algorithms\nThe 3 pillars of good code:\nReadable\n1.\nTime Complexity\n2.\nSpace Complexity\n3.\nWhat skills interviewer is looking for:\nAnalytic Skills - How can you think through problems and analyze things?\nCoding Skills - Do you code well, by writing clean, simple, organized, readable code?\nTechnical knowledge - Do you know the fundamentals of the job you're applying for?\nCommunication skills: Does your personality match the companies’ culture?\nStep By

RAG Pipeline - Data Ingestion to Vector DB

In [8]:
#read all pdfs inside directory
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path 
def process_all_pdfs(storage):
    """processing all pdf files in a directory"""
    all_documents = []
    pdf_dir = Path(storage)

    #find all files recursively i.e check folder and subfolder
    pdf_file = list(pdf_dir.glob("**/*.pdf"))

    for i in pdf_file:
        print(f'processing:{i.name}')
        try:
            loader = PyPDFLoader(str(i))
            documents = loader.load()
            #add source info to metadata
            for doc in documents:
                doc.metadata['source_file'] = i.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f'loaded {len(documents)} pages')
        except Exception as e:
            print(f'Error: {e}')
    print(f'\n Total documents loaded: {len(all_documents)}')
    return all_documents

my_documents = process_all_pdfs('../data')



processing:cheatsheet.pdf
loaded 3 pages

 Total documents loaded: 3


In [9]:
my_documents

[Document(metadata={'producer': 'mPDF 7.1.5', 'creator': 'PyPDF', 'creationdate': "20181019192124+00'00'", 'moddate': "20181019192124+00'00'", 'source': '..\\data\\pdf\\cheatsheet.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'cheatsheet.pdf', 'file_type': 'pdf'}, page_content="Interview Cheat Sheet\nFrom Andrei Neagoie's Master The Coding Interview: Data Structures + Algorithms\nThe 3 pillars of good code:\nReadable1.\nTime Complexity2.\nSpace Complexity3.\nWhat skills interviewer is looking for:\nAnalytic Skills - How can you think through problems and analyze things?\nCoding Skills - Do you code well, by writing clean, simple, organized, readable code?\nTechnical knowledge - Do you know the fundamentals of the job you're applying for?\nCommunication skills: Does your personality match the companies’ culture?\nStep By Step through a problem:\nWhen the interviewer says the question, write down the key points at the top (i.e. sorted1.\narray). Make sure you have 

#TEXT SPLITTING FOR CHUNNKS 

In [10]:
def split_documents(documents, chunk_size=500, chunk_overlap=100):
    """split documents into smaller chunks for better RAG performance"""
    text_splitter =  RecursiveCharacterTextSplitter(
        chunk_size =chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f'Split {len(documents)} documents into {len(split_docs)} chunks')

    if split_docs:
        print(f'\nExample Chunk')
        print(f'Content: {split_docs[0].page_content[:200]}...')
        print(f'Metadata: {split_docs[0].metadata}...')
    return split_docs

In [11]:
chunk = split_documents(my_documents)

Split 3 documents into 16 chunks

Example Chunk
Content: Interview Cheat Sheet
From Andrei Neagoie's Master The Coding Interview: Data Structures + Algorithms
The 3 pillars of good code:
Readable1.
Time Complexity2.
Space Complexity3.
What skills interviewe...
Metadata: {'producer': 'mPDF 7.1.5', 'creator': 'PyPDF', 'creationdate': "20181019192124+00'00'", 'moddate': "20181019192124+00'00'", 'source': '..\\data\\pdf\\cheatsheet.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'cheatsheet.pdf', 'file_type': 'pdf'}...


## #embedding and vectorStoreDb


In [12]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid   #for iding vectors stored in vectordb
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity 

#Embedding Class

In [13]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
       

    def load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingManager()
print("object created")
embedding_manager.load_model()

object created
Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3118.96it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


#vector store 

In [14]:
import os
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 23


In [15]:
chunk

[Document(metadata={'producer': 'mPDF 7.1.5', 'creator': 'PyPDF', 'creationdate': "20181019192124+00'00'", 'moddate': "20181019192124+00'00'", 'source': '..\\data\\pdf\\cheatsheet.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'cheatsheet.pdf', 'file_type': 'pdf'}, page_content="Interview Cheat Sheet\nFrom Andrei Neagoie's Master The Coding Interview: Data Structures + Algorithms\nThe 3 pillars of good code:\nReadable1.\nTime Complexity2.\nSpace Complexity3.\nWhat skills interviewer is looking for:\nAnalytic Skills - How can you think through problems and analyze things?\nCoding Skills - Do you code well, by writing clean, simple, organized, readable code?\nTechnical knowledge - Do you know the fundamentals of the job you're applying for?"),
 Document(metadata={'producer': 'mPDF 7.1.5', 'creator': 'PyPDF', 'creationdate': "20181019192124+00'00'", 'moddate': "20181019192124+00'00'", 'source': '..\\data\\pdf\\cheatsheet.pdf', 'total_pages': 3, 'page': 0, 'page_label

In [16]:
## convert the text to embeddings 
text = [doc.page_content for doc in chunk]


#generate the embeddings using embed manager 
embeds = embedding_manager.generate_embeddings(text)


#store in the vector database 
vectorstore.add_documents(chunk, embeds)

Generating embeddings for 16 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:01<00:00,  1.24s/it]

Generated embeddings with shape: (16, 384)
Adding 16 documents to vector store...
Successfully added 16 documents to vector store
Total documents in collection: 39


# Retriever Pipeline From VectorStore 

In [17]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [18]:
rag_retriever.retrieve('What skills interviewer is looking for')

Retrieving documents for query: 'What skills interviewer is looking for'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.95it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_36fef198_0',
  'content': "Interview Cheat Sheet\nFrom Andrei Neagoie's Master The Coding Interview: Data Structures + Algorithms\nThe 3 pillars of good code:\nReadable1.\nTime Complexity2.\nSpace Complexity3.\nWhat skills interviewer is looking for:\nAnalytic Skills - How can you think through problems and analyze things?\nCoding Skills - Do you code well, by writing clean, simple, organized, readable code?\nTechnical knowledge - Do you know the fundamentals of the job you're applying for?",
  'metadata': {'moddate': "20181019192124+00'00'",
   'content_length': 459,
   'page': 0,
   'producer': 'mPDF 7.1.5',
   'source_file': 'cheatsheet.pdf',
   'total_pages': 3,
   'creationdate': "20181019192124+00'00'",
   'page_label': '1',
   'creator': 'PyPDF',
   'file_type': 'pdf',
   'doc_index': 0,
   'source': '..\\data\\pdf\\cheatsheet.pdf'},
  'similarity_score': 0.25068461894989014,
  'distance': 0.7493153810501099,
  'rank': 1},
 {'id': 'doc_df323c9d_0',
  'content': "Int

## Query retrieval pipeline 
augmented generation 
context + prompt = augmentation 

In [19]:
from langchain_groq import ChatGroq
import os 
from dotenv import load_dotenv
load_dotenv()

False

In [22]:
# Use shared Groq configuration instead of hardcoding the key
from src.pipeline import build_groq_llm

llm = build_groq_llm()

#simple rag function
def rag_simple(query, retriever, llm, top_k=3):
    # retrieve the context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question"

    # generate the answer using Groq LLM
    prompt = f"""Use the following context answer the question concisely.
        Context: {context}
        Question: {query}
        Answer:"""
    response = llm.invoke([prompt])
    return response.content


In [23]:
answer = rag_simple('What skills do interviewer look for', rag_retriever, llm)
print(answer)

Retrieving documents for query: 'What skills do interviewer look for'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 47.88it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


The interviewer looks for:

1. Analytic Skills
2. Coding Skills
3. Technical knowledge
4. Communication skills


In [24]:
#enhanced RAG pipeline 
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What skills do interviewer look for", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What skills do interviewer look for'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 31.52it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: The interviewer looks for the following skills:

1. Analytic Skills
2. Coding Skills
3. Technical knowledge
4. Communication skills
Sources: [{'source': 'cheatsheet.pdf', 'page': 0, 'score': 0.2285243272781372, 'preview': "Interview Cheat Sheet\nFrom Andrei Neagoie's Master The Coding Interview: Data Structures + Algorithms\nThe 3 pillars of good code:\nReadable1.\nTime Complexity2.\nSpace Complexity3.\nWhat skills interviewer is looking for:\nAnalytic Skills - How can you think through problems and analyze things?\nCoding Sk..."}, {'source': 'cheatsheet.pdf', 'page': 0, 'score': 0.22783243656158447, 'preview': "Interview Cheat Sheet\nFrom Andrei Neagoie's Master The Coding Interview: Data Structures + Algorithms\nThe 3 pillars of good code:\nReadable1.\nTime Complexity2.\nSpace Complexity3.\nWhat skills interviewer is looking for:\nAnalytic Skills - How can you think through problems and analyze things?\nCoding Sk..."}, {'source': 'cheatsheet.pdf', 'page': 0, 'score': 0.227832

In [ ]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = "" 
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("What skills do interviewer look for", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary']) 
print("History:", result['history'][-1])

Retrieving documents for query: 'What skills do interviewer look for'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 37.01it/s]

Generated embeddings with shape: (1, 384)


Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Interview Cheat Sheet
From Andrei Neagoie's Master The Coding Interview: Data Structures + Algorithms
The 3 pillars of good code:
Readable1.
Time Complexity2.
Space Complexity3.
What skills interviewer is looking for:
Analytic Skills - How can you think through problems and analyze things?
Coding Skills - Do you code well, by writing clean, simple, organized, readable code?
Technical knowledge - Do you know the fundamentals of the job you're applying for?
Communication skills: Does your personality match the companies’ culture?
Step By Step through a problem:
When the interviewer says the question, write down the key points at the top (i.e. sorted1.
array). Make sure you have all the details. Show how organized you are.
Make sure you double check: What are the inputs? What are the outputs?2.
What is the most important value of the problem? Do you have time, and

In [ ]:
# Refactored RAG usage (backed by src/ modules)

from src.config import get_backend_from_env
from src.pipeline import AdvancedRAGPipeline, build_groq_llm
from src.ingestion import load_text_and_pdfs, split_documents
from src.embeddings import EmbeddingManager
from src.retrieval import ChromaRAGRetriever, TypesenseRAGRetriever
from src.vectorstores import ChromaVectorStore, TypesenseVectorStore

backend = get_backend_from_env("chroma")  # "chroma" or "typesense"
print(f"Using backend: {backend}")

# Load and chunk documents
raw_docs = load_text_and_pdfs()
chunks = split_documents(raw_docs)

if backend == "typesense":
    # Embeddings stored in Typesense
    typesense_store = TypesenseVectorStore()
    typesense_store.build_from_documents(chunks)
    retriever = TypesenseRAGRetriever(typesense_store)
else:
    # Embeddings stored in Chroma
    emb = EmbeddingManager()
    emb.load_model()
    texts = [d.page_content for d in chunks]
    embs = emb.generate_embeddings(texts)

    chroma_store = ChromaVectorStore()
    chroma_store.add_documents(chunks, embs)
    retriever = ChromaRAGRetriever(chroma_store, emb)

llm = build_groq_llm()
rag = AdvancedRAGPipeline(retriever=retriever, llm=llm)

result = rag.query("What skills is the interviewer looking for?", top_k=3, min_score=0.1, summarize=True)
result